# Ingest tables in neo_bank_db in azure into bronze schema

In [0]:
#Imports
from pyspark.sql.functions import current_timestamp

In [0]:
table_mappings = {
    "banking.accounts":"neo_bank.bronze.accounts",
    "banking.branches":"neo_bank.bronze.branches",
    "banking.customers":"neo_bank.bronze.customers",
    "banking.transactions":"neo_bank.bronze.transactions"
}

In [0]:
# Azure SQL Database connection parameters
jdbc_hostname = "neo-bank-server.database.windows.net"
jdbc_port = 1433
jdbc_database = "neo_bank_db"
jdbc_username = "adm"
jdbc_password = "Manjith@9182"  

# Construct JDBC URL
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

# Connection properties
connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [0]:
for source_table, target_table in table_mappings.items():
    # Read data from Azure SQL Database
    df = spark.read.jdbc(
        url=jdbc_url,
        table=source_table,
        properties=connection_properties
    )

    df=df.withColumn("insert_timestamp",current_timestamp())
    
    
    # Write to bronze table
    # Using overwrite mode - change to 'append' if you want to add data incrementally
    (   
        df.write 
        .format("delta") 
        .mode("overwrite") 
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Successfully ingested {df.count()} records from Azure SQL Database to {target_table}")
        